# Face-Detection Confidence by Decade

This notebook extracts the detection-confidence diagnostics from the archive analysis workflow into the thesis workspace. It uses the deduplicated face-detection dataset through 2007 and produces a decade-level summary plus one dual-axis figure: median segmentation confidence and the share of detections below 0.95 confidence.

Execution convention:

Run this notebook from its own directory, `code/scripts`. The repository's VS Code setting `jupyter.notebookFileRoot = ${fileDirname}` makes that the intended execution directory.

Input:

- `../../data/processed/TheEconomistHistoricalArchives-Faces-deduplicated.csv`

Outputs:

- `../../data/processed/face_detection_confidence_by_decade.csv`
- `../../code/output/figures/face_detection_confidence_by_decade.svg`


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
import seaborn as sns

pd.options.display.max_columns = 80
pd.options.display.max_colwidth = 140
sns.set_theme(style="whitegrid", context="notebook")

LOW_CONFIDENCE_THRESHOLD = 0.95
MAX_YEAR_CUTOFF = 2007

# Paths are relative to this notebook's directory: code/scripts.
deduplicated_csv = Path("../../data/processed/TheEconomistHistoricalArchives-Faces-deduplicated.csv")
summary_csv = Path("../../data/processed/face_detection_confidence_by_decade.csv")
figure_svg = Path("../../code/output/figures/face_detection_confidence_by_decade.svg")

expected_columns = [
    "Filename",
    "Bounding Box relative X1",
    "Bounding Box relative Y1",
    "Bounding Box relative X2",
    "Bounding Box relative Y2",
    "Segmentation confidence score",
    "Size relative",
    "Age",
    "Gender",
]

assert deduplicated_csv.exists(), f"Missing deduplicated face CSV: {deduplicated_csv}"
summary_csv.parent.mkdir(parents=True, exist_ok=True)
figure_svg.parent.mkdir(parents=True, exist_ok=True)

print(f"Face CSV:      {deduplicated_csv}")
print(f"Summary CSV:   {summary_csv}")
print(f"Figure output: {figure_svg}")


## Load and Validate Data

The deduplicated CSV is the face-level input. Each row is one retained face detection. The filename prefix identifies the issue year, which is used to assign detections to decades.


In [ ]:
faces = pd.read_csv(deduplicated_csv, dtype={"Filename": "string"})

assert list(faces.columns) == expected_columns, {
    "expected": expected_columns,
    "actual": list(faces.columns),
}
assert len(faces) > 0, "The deduplicated face CSV is empty."

faces = faces.copy()
faces["confidence"] = pd.to_numeric(faces["Segmentation confidence score"], errors="raise")

filename_parts = faces["Filename"].str.extract(r"^(?P<issue_year>\d{4})-\d{4}-")
parse_failures = int(filename_parts["issue_year"].isna().sum())
assert parse_failures == 0, f"Could not parse issue year from {parse_failures:,} filenames."

faces["issue_year"] = pd.to_numeric(filename_parts["issue_year"], errors="raise").astype("int64")
faces = faces.loc[faces["issue_year"] <= MAX_YEAR_CUTOFF].copy()
assert len(faces) > 0, f"No face rows remain after filtering through {MAX_YEAR_CUTOFF}."
assert int(faces["issue_year"].max()) <= MAX_YEAR_CUTOFF

faces["decade_start"] = (faces["issue_year"] // 10) * 10
faces["decade"] = faces["decade_start"].astype("string") + "s"

assert faces["confidence"].between(0, 1).all(), "Confidence values should be probabilities in [0, 1]."
assert faces["issue_year"].between(1800, 2100).all(), "Parsed issue years are outside the expected archive range."

pd.Series(
    {
        "deduplicated_rows_through_2007": len(faces),
        "first_issue_year": int(faces["issue_year"].min()),
        "last_issue_year": int(faces["issue_year"].max()),
        "decades": faces["decade_start"].nunique(),
        "median_confidence": faces["confidence"].median(),
        "share_below_0_95": (faces["confidence"] < LOW_CONFIDENCE_THRESHOLD).mean(),
    },
    name="value",
).to_frame()


## Summarize Confidence by Decade

The summary keeps one row per issue decade through 2007. `median_confidence` is the median segmentation confidence among retained face detections. `share_confidence_below_0_95` is the proportion of retained detections below the 0.95 threshold requested for the thesis diagnostic.


In [ ]:
confidence_summary = (
    faces.groupby(["decade_start", "decade"], as_index=False)
    .agg(
        detected_faces=("Filename", "size"),
        median_confidence=("confidence", "median"),
        detections_below_0_95=("confidence", lambda values: int((values < LOW_CONFIDENCE_THRESHOLD).sum())),
        min_confidence=("confidence", "min"),
        max_confidence=("confidence", "max"),
        first_issue_year=("issue_year", "min"),
        last_issue_year=("issue_year", "max"),
    )
    .sort_values("decade_start")
    .reset_index(drop=True)
)
confidence_summary["share_confidence_below_0_95"] = (
    confidence_summary["detections_below_0_95"] / confidence_summary["detected_faces"]
)

confidence_summary = confidence_summary[
    [
        "decade_start",
        "decade",
        "detected_faces",
        "median_confidence",
        "detections_below_0_95",
        "share_confidence_below_0_95",
        "min_confidence",
        "max_confidence",
        "first_issue_year",
        "last_issue_year",
    ]
]

assert confidence_summary["decade_start"].is_monotonic_increasing
assert confidence_summary["detected_faces"].gt(0).all()
assert confidence_summary["median_confidence"].between(0, 1).all()
assert confidence_summary["share_confidence_below_0_95"].between(0, 1).all()
assert int(confidence_summary["detected_faces"].sum()) == len(faces)

confidence_summary


## Plot Confidence Diagnostics

The figure combines both confidence diagnostics in one dual-axis plot. Both metrics are probabilities, so the two y-axes use the same 0.2-to-1.04 scale and aligned horizontal gridlines: the left axis labels median confidence and the right axis labels the share of detections below 0.95. Ticks stop at 1.00/100%, leaving a small amount of visual headroom above the maximum valid value.


In [ ]:
plot_data = confidence_summary.copy()
x = np.arange(len(plot_data))

median_color = "#4c78a8"
share_color = "#e45756"

fig, ax_median = plt.subplots(figsize=(12, 5.8), constrained_layout=True)
ax_share = ax_median.twinx()
probability_ticks = np.linspace(0.2, 1.0, 5)

median_line = ax_median.plot(
    x,
    plot_data["median_confidence"],
    color=median_color,
    marker="o",
    linewidth=2.2,
    markersize=5,
)
share_line = ax_share.plot(
    x,
    plot_data["share_confidence_below_0_95"],
    color=share_color,
    marker="s",
    linewidth=2.2,
    markersize=5,
)

ax_median.set_title("Face-detection confidence by issue decade", fontsize=14, pad=12)
ax_median.set_xlabel("Issue decade")
ax_median.set_ylabel("Median confidence")
ax_median.set_xticks(x)
ax_median.set_xticklabels(plot_data["decade"], rotation=45, ha="right")
ax_median.set_ylim(0.2, 1.04)
ax_median.set_yticks(probability_ticks)
ax_median.yaxis.set_major_formatter(lambda value, position: f"{value:.2f}")
ax_median.grid(True, axis="y", alpha=0.35)
ax_median.grid(True, axis="x", alpha=0.10)

ax_share.set_ylabel("Share of detections")
ax_share.set_ylim(0.2, 1.04)
ax_share.set_yticks(probability_ticks)
ax_share.yaxis.set_major_formatter(PercentFormatter(1.0))
ax_share.grid(False)

handles = [median_line[0], share_line[0]]
labels = ["Median confidence", "Share below 0.95"]
ax_median.legend(handles, labels, loc="lower left", frameon=True, framealpha=0.95)

sns.despine(ax=ax_median, right=False)
fig.savefig(figure_svg, format="svg", bbox_inches="tight")
plt.show()

figure_svg


## Write and Verify Outputs

The CSV and SVG are derived analysis artifacts. They can be regenerated by re-running this notebook after the deduplicated face dataset changes.


In [ ]:
confidence_summary.to_csv(summary_csv, index=False)

reloaded_summary = pd.read_csv(summary_csv)
assert list(reloaded_summary.columns) == list(confidence_summary.columns)
assert len(reloaded_summary) == len(confidence_summary)
assert int(reloaded_summary["detected_faces"].sum()) == len(faces)

assert figure_svg.exists(), f"Missing figure output: {figure_svg}"
assert figure_svg.stat().st_size > 0, f"Empty figure output: {figure_svg}"

pd.Series(
    {
        "summary_csv": str(summary_csv),
        "summary_rows": len(reloaded_summary),
        "figure_svg": str(figure_svg),
        "figure_bytes": figure_svg.stat().st_size,
    },
    name="value",
).to_frame()


## Conclusion

The deduplicated face-detection dataset has complete segmentation confidence values. This notebook records the decade-level median confidence and the share of detections below 0.95 through 2007, then saves the combined figure for thesis use.
